In [1]:
%pip install gymnasium torch numpy matplotlib opencv-python torchsummary

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
import cv2
from torchsummary import summary

In [3]:
SEED = 52

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
gym.utils.seeding.np_random(SEED)

(Generator(PCG64) at 0x27580FF9A80, 52)

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# Hyperparameters

GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_END = 0.1
EPSILON_DECAY = 1000
LEARNING_RATE = 0.0003
BATCH_SIZE = 64
TARGET_UPDATE = 1000
MEMORY_SIZE = 100000
NUM_EPISODES = 300

In [6]:
class DQN(nn.Module):
    """
    Deep Q-Network model definition.

    Args:
        input_shape (int): Dimension of the input state.
        num_actions (int): Number of possible actions.
    """

    def __init__(self, input_shape, num_actions):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_shape, 512)
        self.fc2 = nn.Linear(512, num_actions)

    def forward(self, x):
        """
        Forward pass of the network.

        Args:
            x (torch.Tensor): Input tensor.
        Returns:
            torch.Tensor: Output Q-values for each action.
        """
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return x

In [7]:
class ReplayBuffer:
    """
    Experience replay buffer for storing and sampling transitions.

    Args:
        capacity (int): Maximum number of transitions to store.
    """

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """
        Store a transition in the buffer.

        Args:
            state (np.ndarray): Current state.
            action (int): Action taken.
            reward (float): Reward received.
            next_state (np.ndarray): Next state.
            done (bool): Whether the episode ended.
        """
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        """
        Sample a batch of transitions from the buffer.

        Args:
            batch_size (int): Number of samples to return.
        Returns:
            tuple: Batch of (state, action, reward, next_state, done)
        """
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return (np.array(state), np.array(action), np.array(reward),
                np.array(next_state), np.array(done))

    def __len__(self):
        """
        Return the current size of internal memory.

        Returns:
            int: Number of elements in the buffer.
        """
        return len(self.buffer)

In [8]:
def select_action(state, epsilon, n_actions, online_net, DEVICE=DEVICE):
    """
    Epsilon-greedy action selection.

    Args:
        state (np.ndarray): Current state.
        epsilon (float): Exploration rate.
        n_actions (int): Number of possible actions.
        online_net (DQN): Online Q-network.
        DEVICE (torch.device): Device to use.
    Returns:
        int: Selected action.
    """
    if random.random() > epsilon:
        with torch.no_grad():
            state = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            return online_net(state).argmax(1).item()
    else:
        return random.randrange(n_actions)

In [9]:
def optimize_model(online_dqn, target_dqn, memory: ReplayBuffer, online_optimizer, DEVICE=DEVICE):
    """
    Perform a single optimization step for the online network using DDQN logic.

    Args:
        online_dqn (DQN): Online Q-network.
        target_dqn (DQN): Target Q-network.
        memory (ReplayBuffer): Experience replay buffer.
        online_optimizer (torch.optim.Optimizer): Optimizer for the online network.
        DEVICE (torch.device): Device to use.
    """

    if len(memory) < BATCH_SIZE:
        return
    
    state, action, reward, next_state, done = memory.sample(BATCH_SIZE)

    # Convert to PyTorch Tensors and move them to DEVICE
    state = torch.FloatTensor(state).to(DEVICE)
    action = torch.LongTensor(action).to(DEVICE)
    reward = torch.FloatTensor(reward).to(DEVICE)
    next_state = torch.FloatTensor(next_state).to(DEVICE)
    done = torch.FloatTensor(done).to(DEVICE)

    # Q-value from online network for actions taken
    q_value = online_dqn(state).gather(1, action.unsqueeze(1)).squeeze(1)

    # Best next actions from online network
    next_actions = online_dqn(next_state).argmax(1).unsqueeze(1)

    # Q-value from target network for those actions
    next_q_value = target_dqn(next_state).gather(1, next_actions).squeeze(1)
    expected_q_value = reward + GAMMA * next_q_value * (1 - done)

    loss = (q_value - expected_q_value.detach()).pow(2).mean()

    online_optimizer.zero_grad()
    loss.backward()
    online_optimizer.step()

In [10]:
def ddqn(env, online_dqn, target_dqn, memory, target_update, DEVICE=DEVICE):
    """
    Main training loop for Double DQN agent.

    Args:
        env (gym.Env): The environment.
        online_dqn (DQN): Online Q-network.
        target_dqn (DQN): Target Q-network.
        memory (ReplayBuffer): Experience replay buffer.
        target_update (int): Number of episodes between target network updates.
        DEVICE (torch.device): Device to use.
    """
    num_actions = env.action_space.n

    online_optimizer = optim.Adam(online_dqn.parameters(), lr=LEARNING_RATE)

    epsilon = EPSILON_START
    epsilon_decay = (EPSILON_START - EPSILON_END) / EPSILON_DECAY

    steps_done = 0
    episode_rewards = []

    for episode in range(NUM_EPISODES):
        state, _ = env.reset()
        total_reward = 0
        for t in range(1000):
            # Epsilon-greedy action selection
            action = select_action(state, epsilon, num_actions, online_dqn, DEVICE)
            next_state, reward, done, _, _ = env.step(action)

            if done and reward <= 0:
                reward = -1
            
            total_reward += reward

            # Store transition in replay buffer
            memory.push(state, action, reward, next_state, done)

            state = next_state

            # Optimize the online network
            optimize_model(online_dqn, target_dqn, memory, online_optimizer)

            if done:
                break

            epsilon = max(EPSILON_END, epsilon - epsilon_decay)
            steps_done += 1

        # Update target network
        if episode % target_update == 0:
            target_dqn.load_state_dict(online_dqn.state_dict())

        episode_rewards.append(total_reward)
        print(f"Episode {episode + 1}, Total reward: {total_reward}")

In [ ]:
# Unified environment and network initialization
# Only keep this cell for setup, remove duplicates

env = gym.make('CartPole-v1')
input_shape = 4
num_actions = env.action_space.n  # type: ignore

online_net = DQN(input_shape, num_actions).to(DEVICE)
target_net = DQN(input_shape, num_actions).to(DEVICE)
memory = ReplayBuffer(MEMORY_SIZE)

ddqn(env, online_net, target_net, memory, TARGET_UPDATE)

env.close()

Episode 1, Total reward: 25.0
Episode 2, Total reward: 12.0
Episode 3, Total reward: 13.0
Episode 4, Total reward: 27.0
Episode 5, Total reward: 13.0
Episode 6, Total reward: 22.0
Episode 7, Total reward: 9.0
Episode 8, Total reward: 16.0
Episode 9, Total reward: 30.0
Episode 10, Total reward: 14.0
Episode 11, Total reward: 27.0
Episode 12, Total reward: 36.0
Episode 13, Total reward: 13.0
Episode 14, Total reward: 15.0
Episode 15, Total reward: 31.0
Episode 16, Total reward: 45.0
Episode 11, Total reward: 27.0
Episode 12, Total reward: 36.0
Episode 13, Total reward: 13.0
Episode 14, Total reward: 15.0
Episode 15, Total reward: 31.0
Episode 16, Total reward: 45.0
Episode 17, Total reward: 33.0
Episode 18, Total reward: 13.0
Episode 19, Total reward: 17.0
Episode 20, Total reward: 14.0
Episode 21, Total reward: 25.0
Episode 22, Total reward: 46.0
Episode 23, Total reward: 12.0
Episode 17, Total reward: 33.0
Episode 18, Total reward: 13.0
Episode 19, Total reward: 17.0
Episode 20, Total 